# HWD Susenas (Colab) - MCAR 10% + Rule Engine (Kualitatif)

Versi Colab. Kode dan data dibaca dari Google Drive. Isi sama dengan versi Kaggle: preprocessing, training (checkpoint + resume), inferensi, evaluasi akurasi, dan demonstrasi kualitatif survey rule engine.

## Cell 1 — Install Dependencies

In [ ]:
!pip install PyWavelets properscoring --quiet

## Cell 2 — Setup Path & File HWD (Kaggle)

In [ ]:
import os, sys, shutil, glob, time
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Skripsi/HWD'
DATA_DIR    = f'{PROJECT_DIR}/Data'
CKPT_DIR    = f'{PROJECT_DIR}/Checkpoints'
HWD_DIR     = PROJECT_DIR

for d in [DATA_DIR, CKPT_DIR, f'{DATA_DIR}/mask/susenas']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, HWD_DIR)
os.chdir(HWD_DIR)
print('Working dir:', os.getcwd())
print('Isi folder:', [f for f in os.listdir(HWD_DIR) if f.endswith('.py') or f in ('models','layers')])


Mounted at /content/drive
Working dir: /content/drive/MyDrive/Skripsi/HWD
Isi folder: ['utils.py', 'incremental.py', 'layers', 'models', 'postprocess.py', 'generate_missing.py', 'train.py', 'dataset.py', 'survey_rules.py']


## Cell 3 — Import

In [ ]:
import numpy as np
import pandas as pd
import torch
import time, importlib.util
import properscoring as ps
warnings_shown = False
np.random.seed(1); torch.manual_seed(1)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# Muat survey_rules.py
spec = importlib.util.spec_from_file_location('sr', f'{HWD_DIR}/survey_rules.py')
sr = importlib.util.module_from_spec(spec); spec.loader.exec_module(sr)
print('survey_rules dimuat:', len(sr.SUSENAS_SKIP_RULES), 'rule')

Device: cuda
survey_rules dimuat: 2 rule


## Cell 4 — Preprocessing Data Susenas

Data Susenas (sudah tergabung antar tahun via kode_sampel) diproses:
- Kolom numerik yang masuk model: R101, R102, R105, R301, R705, R706, R707, food, nonfood, expend, kapita
- kode_sampel, tahun, dan R703_A DIKELUARKAN dari fitur model
- R703_A tetap disimpan terpisah karena dibutuhkan survey rule engine
- Normalisasi Z-score per kolom; nilai hilang ditandai sentinel -200

In [ ]:
# ── Muat data Susenas dari Google Drive ──────────────────────────────────────
import glob, os
import numpy as np, pandas as pd

cands = (glob.glob(f'{DATA_DIR}/*usenas*.csv') +
         glob.glob(f'{DATA_DIR}/*usenas*.xlsx') +
         glob.glob(f'{PROJECT_DIR}/**/*usenas*.csv',  recursive=True) +
         glob.glob(f'{PROJECT_DIR}/**/*usenas*.xlsx', recursive=True))
assert cands, f'File susenas tidak ditemukan di {DATA_DIR}.'
FSUS = cands[0]
print('File susenas ditemukan:', FSUS)

df_raw = pd.read_excel(FSUS) if FSUS.lower().endswith(('.xlsx','.xls')) else pd.read_csv(FSUS)
print('Data mentah:', df_raw.shape)
print('Kolom:', list(df_raw.columns))

# ── Kolom fitur model ────────────────────────────────────────────────────────
FEAT_COLS = ['R101','R102','R105','R301','R705','R706','R707',
             'food','nonfood','expend','kapita']
colmap = {c.lower(): c for c in df_raw.columns}
def col(name):
    return colmap.get(name.lower(), name)
FEAT_COLS = [col(c) for c in FEAT_COLS]

# ── Kolom yang DIKECUALIKAN dari penghapusan outlier ─────────────────────────
# Identitas & kategorik: R101 provinsi, R102 kab/kota, R105 klasifikasi wilayah,
# R301 jumlah ART, R705/R706/R707 ketenagakerjaan.
# Hanya 4 variabel pengeluaran yang dicek outliernya.
SKIP_OUTLIER_IDX = [0, 1, 2, 3, 4, 5, 6]

print('\nUrutan FEAT_COLS:')
for i, c in enumerate(FEAT_COLS):
    tag = 'skip' if i in SKIP_OUTLIER_IDX else 'CEK OUTLIER'
    print(f'  {i:>2}  {c:<12} → {tag}')

# ── Kolom pendukung ──────────────────────────────────────────────────────────
R703_COL      = col('R703_A')
ID_OUT_COLS   = [c for c in [col('kode_sampel'), col('tahun')] if c in df_raw.columns]
_support_cols = (ID_OUT_COLS
                 + ([R703_COL] if R703_COL in df_raw.columns else [])
                 + FEAT_COLS)
df_support    = df_raw[_support_cols].copy()
print('\nKolom identitas output:', ID_OUT_COLS)
if R703_COL not in df_raw.columns:
    print(f'Catatan: kolom {R703_COL} tidak ditemukan — dilewati')

# ── Matriks fitur numerik ────────────────────────────────────────────────────
X = df_raw[FEAT_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
n_rows, n_feat = X.shape
print(f'Fitur model: {n_feat} kolom, {n_rows} baris')

# ═══════════════════════════════════════════════════════════════════════════
#  PENANGANAN OUTLIER
#  True  → operational mode (outlier dihapus sebelum normalisasi)
#  False → benchmark mode  (outlier dibiarkan)
# ═══════════════════════════════════════════════════════════════════════════
REMOVE_OUTLIER = True

n_before = int(np.isnan(X).sum())
print(f'\nMissing alami sebelum penanganan : {n_before} sel '
      f'({n_before/(n_rows*n_feat)*100:.2f}%)')

outlier_detail = {}
if REMOVE_OUTLIER:
    n_out = 0
    for j in range(n_feat):
        if j in SKIP_OUTLIER_IDX:
            continue
        colv  = X[:, j]
        valid = colv[~np.isnan(colv)]
        if len(valid) < 10:
            continue
        q1, q3 = np.percentile(valid, 25), np.percentile(valid, 75)
        iqr    = q3 - q1
        lo, hi = q1 - 3.0*iqr, q3 + 3.0*iqr
        if q1 >= 0:                       # pengeluaran non-negatif
            lo = max(lo, 0.0)
        mask_out = (~np.isnan(colv)) & ((colv < lo) | (colv > hi))
        k = int(mask_out.sum())
        if k > 0:
            outlier_detail[FEAT_COLS[j]] = k
        n_out += k
        X[mask_out, j] = np.nan
    print(f'Outlier 3xIQR dihapus            : {n_out} sel')
    for k_, v_ in outlier_detail.items():
        print(f'   - {k_:<12}: {v_} sel')
else:
    print('Outlier TIDAK dihapus (benchmark mode)')

n_after = int(np.isnan(X).sum())
print(f'Total missing setelah penanganan : {n_after} sel '
      f'({n_after/(n_rows*n_feat)*100:.2f}%)')

# ── Normalisasi Z-score ──────────────────────────────────────────────────────
# Mean & std dihitung SETELAH outlier dihapus agar skala tidak tertarik ekstrem
means = np.nanmean(X, axis=0)
stds  = np.nanstd(X, axis=0); stds[stds == 0] = 1.0
Xn = (X - means) / stds
Xn[np.isnan(Xn)] = -200.0

MODE = 'operational' if REMOVE_OUTLIER else 'benchmark'
print(f'\nNormalisasi selesai (mode: {MODE})')
print(f'Sentinel -200 pada posisi missing : {int((Xn==-200).sum())} sel')

# ── Simpan means/stds untuk denormalisasi nanti ──────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
np.save(f'{DATA_DIR}/susenas_means_{MODE}.npy', means)
np.save(f'{DATA_DIR}/susenas_stds_{MODE}.npy',  stds)
print(f'means/stds tersimpan di {DATA_DIR}/')

File susenas ditemukan: /content/drive/MyDrive/Skripsi/HWD/Data/Susenas.csv


/tmp/ipykernel_1488/2150176041.py:13: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_excel(FSUS) if FSUS.lower().endswith(('.xlsx','.xls')) else pd.read_csv(FSUS)


Data mentah: (1028741, 14)
Kolom: ['kode_sampel', 'Tahun', 'R101', 'R102', 'R105', 'R301', 'R703_A', 'R705', 'R706', 'R707', 'FOOD', 'NONFOOD', 'EXPEND', 'KAPITA']

Urutan FEAT_COLS:
   0  R101         → skip
   1  R102         → skip
   2  R105         → skip
   3  R301         → skip
   4  R705         → skip
   5  R706         → skip
   6  R707         → skip
   7  FOOD         → CEK OUTLIER
   8  NONFOOD      → CEK OUTLIER
   9  EXPEND       → CEK OUTLIER
  10  KAPITA       → CEK OUTLIER

Kolom identitas output: ['kode_sampel', 'Tahun']
Fitur model: 11 kolom, 1028741 baris

Missing alami sebelum penanganan : 108245 sel (0.96%)
Outlier 3xIQR dihapus            : 104743 sel
   - FOOD        : 9873 sel
   - NONFOOD     : 39483 sel
   - EXPEND      : 24166 sel
   - KAPITA      : 31221 sel
Total missing setelah penanganan : 212988 sel (1.88%)

Normalisasi selesai (mode: operational)
Sentinel -200 pada posisi missing : 212988 sel
means/stds tersimpan di /content/drive/MyDrive/Skripsi/HWD

## Cell 5 — Windowing & Mask MCAR 10%

Data disusun menjadi window sepanjang seq_len. Baris diperlakukan independen (tanpa
asumsi keterkaitan temporal antarbaris); windowing hanya menata data agar sesuai bentuk
masukan model. Mask MCAR menyembunyikan 10% posisi teramati secara acak seragam sebagai
artificial missing untuk evaluasi.

In [ ]:
SEQ_LEN      = 48
MISSING_RATE = 0.1
SEED         = 1
MECHANISM    = 'mar'       # ← 'mcar' | 'mar' | 'block'

ID_COLS_IDX = [0, 1]                       # R101 provinsi, R102 kab/kota
MAR_COL     = FEAT_COLS.index(col('food')) # pemicu MAR: pengeluaran makanan
BLOCK_LEN   = 4                            # panjang blok untuk block missing

# ── Reshape ke [N_win, seq_len, n_feat] ──────────────────────────────────────
n_ok = (n_rows // SEQ_LEN) * SEQ_LEN
data = Xn[:n_ok].reshape(-1, SEQ_LEN, n_feat)
gt_mask = (data != -200).astype(np.float32)
print('Bentuk data:', data.shape)


def make_mask(data, gt_mask, mechanism, missing_rate, seed,
              id_cols_idx, mar_col=None, block_len=4):
    rng = np.random.default_rng(seed)
    N, L, K = data.shape
    cond_mask = gt_mask.copy()

    obs_idx = np.argwhere(gt_mask == 1)
    obs_idx = obs_idx[~np.isin(obs_idx[:, 2], id_cols_idx)]
    n_hide  = int(len(obs_idx) * missing_rate)

    if mechanism == 'mcar':
        pick = obs_idx[rng.choice(len(obs_idx), n_hide, replace=False)]
        for (a, b, c) in pick:
            cond_mask[a, b, c] = 0.0

    elif mechanism == 'mar':
        # Rata-rata food per window HANYA dari posisi teramati (perbaikan bug)
        fval = data[:, :, mar_col]
        fobs = gt_mask[:, :, mar_col]
        fsum = (fval * fobs).sum(axis=1)
        fcnt = fobs.sum(axis=1)
        fmean = np.where(fcnt > 0, fsum / np.maximum(fcnt, 1), np.nan)
        fmean[np.isnan(fmean)] = np.nanmin(fmean)

        rank  = (np.argsort(np.argsort(fmean)) + 1).astype(np.float64)
        w_win = rank / rank.sum()

        weight = np.repeat(w_win, L * K).reshape(N, L, K) * gt_mask
        weight[:, :, id_cols_idx] = 0.0
        weight = weight.flatten()
        weight = weight / weight.sum()

        u      = rng.random(weight.shape)
        keys   = -np.log(u + 1e-12) / (weight + 1e-12)
        chosen = np.argpartition(keys, n_hide)[:n_hide]

        flat = cond_mask.flatten()
        flat[chosen] = 0.0
        cond_mask = flat.reshape(N, L, K)

    elif mechanism == 'block':
        maskable = [c for c in range(K) if c not in id_cols_idx]
        hidden, guard = 0, 0
        while hidden < n_hide and guard < n_hide * 50:
            guard += 1
            w = rng.integers(0, N)
            c = maskable[rng.integers(0, len(maskable))]
            s = rng.integers(0, max(1, L - block_len + 1))
            for t in range(s, min(s + block_len, L)):
                if gt_mask[w, t, c] == 1 and cond_mask[w, t, c] == 1:
                    cond_mask[w, t, c] = 0.0
                    hidden += 1
    else:
        raise ValueError(f'mechanism tidak dikenal: {mechanism}')

    return cond_mask


cond_mask = make_mask(data, gt_mask, MECHANISM, MISSING_RATE, SEED,
                      ID_COLS_IDX, mar_col=MAR_COL, block_len=BLOCK_LEN)

eval_target = ((gt_mask == 1) & (cond_mask == 0))
n_target    = int(eval_target.sum())
actual      = n_target / max(int(gt_mask.sum()), 1)

print(f'\nMekanisme            : {MECHANISM.upper()}')
print(f'Target missing rate  : {MISSING_RATE:.2f}')
print(f'Posisi disembunyikan : {n_target}')
print(f'Actual missing rate  : {actual:.4f}')

Bentuk data: (21432, 48, 11)

Mekanisme            : MAR
Target missing rate  : 0.10
Posisi disembunyikan : 904563
Actual missing rate  : 0.0815


## Cell 6 — Konfigurasi Model HWD

In [ ]:
from types import SimpleNamespace

ABLATION = None        # 'no_wavelet' | 'no_crosslevel' | None
ARCH_V2  = False       # True = arsitektur v2 (tiga perbaikan penyaluran sinyal)
WAVELET  = 'db2'       # db2 agar 3 tingkat tercapai pada seq_len 48
LEVELS   = 3

cfg = SimpleNamespace(
    seq_len=SEQ_LEN, enc_in=n_feat, c_out=n_feat,
    d_model=128, e_layers=4, nheads=8, channel=128, proj_t=128,
    residual_layers=4, timeemb=128, featureemb=16,
    diffusion_step_num=50, schedule='quad', beta_start=1e-4, beta_end=0.2,
    epoch_diff=100, learning_rate_diff=1e-3,
    mask_ratio_ssl=0.2, avg_mask_len_ssl=3,
    wavelet=WAVELET, levels=LEVELS, batch=32,
    device=DEVICE,
    missing_rate=MISSING_RATE,
    seed=SEED,
    n_samples=10,
    mechanism=MECHANISM,
    ablation=ABLATION,
    arch_v2=ARCH_V2,
)

ABL  = ABLATION if ABLATION else 'full'
ARCH = 'v2' if ARCH_V2 else 'v1'

print(f'Config siap  |  enc_in = {cfg.enc_in}')
print(f'Mekanisme    : {cfg.mechanism}')
print(f'Ablation     : {ABL}  |  Arch: {ARCH}')
print(f'Missing rate : {cfg.missing_rate}')
print(f'Wavelet      : {cfg.wavelet}, levels = {cfg.levels}')
print(f'Mode praproses: {MODE}')
print(f'Diffusion    : T={cfg.diffusion_step_num}, batch={cfg.batch}, epoch={cfg.epoch_diff}')

Config siap  |  enc_in = 11
Mekanisme    : mar
Ablation     : full  |  Arch: v1
Missing rate : 0.1
Wavelet      : db2, levels = 3
Mode praproses: operational
Diffusion    : T=50, batch=32, epoch=100


## Cell 7 — Dataset Loader (inline)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SusenasDataset(Dataset):
    def __init__(self, data, cond_mask, gt_mask):
        self.data      = torch.from_numpy(data).float()
        self.cond_mask = torch.from_numpy(cond_mask).float()
        self.gt_mask   = torch.from_numpy(gt_mask).float()
        self.tp        = torch.arange(data.shape[1], dtype=torch.float32)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, i):
        return (self.data[i], self.cond_mask[i], self.tp, self.gt_mask[i])


ds = SusenasDataset(data, cond_mask, gt_mask)

# Batch inferensi lebih besar untuk mempercepat; tidak mengubah hasil.
# Turunkan bila terjadi OOM.
INFER_BATCH  = 32
train_loader = DataLoader(ds, batch_size=cfg.batch, shuffle=True)
test_loader  = DataLoader(ds, batch_size=INFER_BATCH, shuffle=False)

print(f'Dataset      : {len(ds)} window')
print(f'Mekanisme    : {MECHANISM} (rate {MISSING_RATE})  |  Mode: {MODE}')
print(f'Wavelet      : {cfg.wavelet} L={cfg.levels}  |  Abl: {ABL}  |  Arch: {ARCH}')
print(f'Batch train  : {cfg.batch} | Batch inferensi: {INFER_BATCH}')

Dataset      : 21432 window
Mekanisme    : mar (rate 0.1)  |  Mode: operational
Wavelet      : db2 L=3  |  Abl: full  |  Arch: v1
Batch train  : 32 | Batch inferensi: 32


## Cell 8 — Training HWD (sekali)

Model dilatih sekali. Selama pelatihan, fungsi loss hanya dihitung pada posisi target
(cond_mask==0 & gt_mask==1), sehingga model belajar merekonstruksi nilai yang disembunyikan.

In [ ]:
from models import main_model
from torch import optim
import os, glob, math, time

model     = main_model.HWD(cfg).to(cfg.device)
optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate_diff, weight_decay=1e-6)
p1, p2    = int(0.75*cfg.epoch_diff), int(0.90*cfg.epoch_diff)
sched     = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[p1, p2], gamma=0.1)

# Nama checkpoint memuat seluruh konfigurasi agar tidak saling menimpa
stem        = f'susenas_{MECHANISM}_mr{MISSING_RATE}_{ABL}_{ARCH}_{WAVELET}_{MODE}'
CKPT_PATH   = f'{CKPT_DIR}/hwd_{stem}.pt'
losses      = []
start_epoch = 0

FORCE_RETRAIN = False   # True = abaikan checkpoint lama, latih dari awal

# Colab: checkpoint persisten di Drive, cukup cek satu lokasi
_found = None if FORCE_RETRAIN else (CKPT_PATH if os.path.exists(CKPT_PATH) else None)

if _found:
    ckpt = torch.load(_found, map_location=cfg.device, weights_only=False)
    try:
        model.load_state_dict(ckpt['model'])
        if 'optimizer' in ckpt: optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt: sched.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt.get('epoch', cfg.epoch_diff-1) + 1
        losses      = ckpt.get('losses', [])
        print(f'>> Checkpoint dimuat: {_found}')
        if start_epoch >= cfg.epoch_diff:
            print(f'>> Model sudah terlatih penuh ({start_epoch} epoch). Siap inferensi.')
        else:
            print(f'>> Lanjut training dari epoch {start_epoch}/{cfg.epoch_diff}')
    except RuntimeError:
        print(f">> Checkpoint TIDAK COCOK dengan arsitektur sekarang "
              f"(ckpt wavelet={ckpt.get('wavelet','db4')}, "
              f"levels={ckpt.get('levels','?')}) — training dari awal.")
        model     = main_model.HWD(cfg).to(cfg.device)
        optimizer = optim.Adam(model.parameters(),
                               lr=cfg.learning_rate_diff, weight_decay=1e-6)
        sched     = torch.optim.lr_scheduler.MultiStepLR(
                        optimizer, milestones=[p1, p2], gamma=0.1)
        losses, start_epoch = [], 0
else:
    print(f'>> Tidak ada checkpoint [{MECHANISM} | {ABL} | {ARCH}]. Training dari awal.')

# ── Loop training ────────────────────────────────────────────────────────────
for epoch in range(start_epoch, cfg.epoch_diff):
    model.train(); ep = []; t0 = time.time()
    for obs_d, obs_m, obs_tp, gt_m in train_loader:
        obs_d = obs_d.to(cfg.device); obs_m = obs_m.to(cfg.device); gt_m = gt_m.to(cfg.device)
        optimizer.zero_grad()
        loss = model(obs_d, obs_m, obs_tp.to(cfg.device), gt_m)
        loss.backward(); optimizer.step(); ep.append(loss.item())
    sched.step(); losses.append(np.mean(ep))

    saved = (epoch % 25 == 0) or (epoch == cfg.epoch_diff - 1)
    if saved:
        os.makedirs(CKPT_DIR, exist_ok=True)
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': sched.state_dict(),
                    'losses': losses,
                    'mechanism': MECHANISM,
                    'missing_rate': MISSING_RATE,
                    'ablation': ABLATION,
                    'arch_v2': ARCH_V2,
                    'wavelet': WAVELET,
                    'levels': LEVELS,
                    'mode': MODE}, CKPT_PATH)

    if epoch % 5 == 0 or epoch == cfg.epoch_diff - 1:
        tag = ' | ckpt tersimpan' if saved else ''
        print(f'Epoch {epoch+1:>4}/{cfg.epoch_diff} | {time.time()-t0:5.1f}s '
              f'| loss {np.mean(ep):.6f}{tag}')

print(f'\nSelesai [{MECHANISM} | {ABL} | {ARCH} | {WAVELET} | {MODE}]')
print(f'Checkpoint: {CKPT_PATH}')

[diff_HWD] levels aktif: 3 (wavelet=db2)


/content/drive/MyDrive/Skripsi/HWD/models/ts_transformer.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, configs.e_layers)


>> Tidak ada checkpoint [mar | full | v1]. Training dari awal.
Epoch    1/100 | 168.5s | loss 0.426596 | ckpt tersimpan
Epoch    6/100 | 169.2s | loss 0.268937
Epoch   11/100 | 168.7s | loss 0.252905
Epoch   16/100 | 168.9s | loss 0.247448
Epoch   21/100 | 169.2s | loss 0.242699
Epoch   26/100 | 169.8s | loss 0.241604 | ckpt tersimpan
Epoch   31/100 | 170.5s | loss 0.237301
Epoch   36/100 | 169.4s | loss 0.235907
Epoch   41/100 | 168.9s | loss 0.232016
Epoch   46/100 | 168.9s | loss 0.232303
Epoch   51/100 | 170.0s | loss 0.233688 | ckpt tersimpan
Epoch   56/100 | 169.2s | loss 0.231371
Epoch   61/100 | 168.7s | loss 0.230136
Epoch   66/100 | 170.0s | loss 0.229712
Epoch   71/100 | 168.9s | loss 0.228019
Epoch   76/100 | 169.3s | loss 0.219747 | ckpt tersimpan
Epoch   81/100 | 169.3s | loss 0.216085
Epoch   86/100 | 168.7s | loss 0.215623
Epoch   91/100 | 168.8s | loss 0.214648
Epoch   96/100 | 169.4s | loss 0.214854
Epoch  100/100 | 169.4s | loss 0.213230 | ckpt tersimpan

Selesai [ma

## Cell 9 — Inferensi & Imputasi

Model menghasilkan sampel imputasi pada posisi target. Median dari sampel dipakai sebagai
taksiran titik. Hasil ini adalah keluaran mentah model, sebelum survey rule engine.

In [ ]:
model.eval()
import math, time
n_test_batch = math.ceil(len(ds) / INFER_BATCH)

def run_inference(loader):
    imputed = np.zeros((data.shape[0], data.shape[2], data.shape[1]))  # [N, K, L]
    slist = []
    idx = 0; t_start = time.time()
    with torch.no_grad():
        for bi, (obs_d, obs_m, obs_tp, gt_m) in enumerate(loader):
            obs_d=obs_d.to(cfg.device); obs_m=obs_m.to(cfg.device)
            obs_tp=obs_tp.to(cfg.device); gt_m=gt_m.to(cfg.device)
            t0=time.time()
            samples, _, _, _, _ = model.evaluate(obs_d, obs_m, obs_tp, gt_m,
                                                 n_samples=cfg.n_samples)   # [B,n,K,L]
            med = samples.median(dim=1).values                             # [B,K,L]
            b = med.shape[0]
            imputed[idx:idx+b] = med.cpu().numpy()
            slist.append(samples.cpu().numpy())
            idx += b
            if bi < 5 or bi % 10 == 0 or bi == n_test_batch-1:
                el = time.time()-t0
                eta = (n_test_batch-bi-1)*el/60
                print(f'  batch {bi+1}/{n_test_batch} | {el:.1f}s/batch | ETA ~{eta:.0f} min')
    return imputed, np.concatenate(slist, axis=0)

try:
    imputed_median, samples_all = run_inference(test_loader)
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('>> OOM terdeteksi, turunkan batch inferensi ke 8 dan coba lagi...')
        torch.cuda.empty_cache()
        test_loader = DataLoader(ds, batch_size=8, shuffle=False)
        n_test_batch = math.ceil(len(ds)/8)
        imputed_median, samples_all = run_inference(test_loader)
    else:
        raise

# Ubah ke [N, L, K] agar konsisten dengan data asli
imputed_median = imputed_median.transpose(0, 2, 1)          # [N, L, K]
samples_all = samples_all.transpose(0, 1, 3, 2)             # [N, n, L, K]
print('Inferensi selesai. Bentuk imputasi:', imputed_median.shape)


## Cell 10 — Evaluasi Akurasi (MAE, RMSE, CRPS)

In [ ]:
# Denormalisasi ke skala asli untuk MAE/RMSE yang bermakna? -> evaluasi pd skala ternormalisasi (konsisten benchmark)
tgt = eval_target   # [N,L,K]
true_vals = data[tgt]
pred_vals = imputed_median[tgt]

mae  = np.mean(np.abs(pred_vals - true_vals))
rmse = np.sqrt(np.mean((pred_vals - true_vals)**2))

# CRPS pakai sampel penuh di posisi target
crps_list=[]
Nw,ns,Lw,Kw = samples_all.shape
ti = np.argwhere(tgt)
for (a,b,c) in ti:
    crps_list.append(ps.crps_ensemble(data[a,b,c], samples_all[a,:,b,c]))
crps = np.mean(crps_list) if crps_list else float('nan')

print('=== AKURASI (skala ternormalisasi) ===')
print(f'  MAE  = {mae:.4f}')
print(f'  RMSE = {rmse:.4f}')
print(f'  CRPS = {crps:.4f}')

## Cell 11 — Rekonstruksi Data Terimputasi (skala asli)

Nilai imputasi didenormalisasi kembali ke skala asli, lalu disusun ulang menjadi tabel
rumah tangga dengan kolom fitur semula. Nilai teramati dipertahankan; hanya posisi
artificial/natural missing yang diisi hasil model.

In [ ]:
# gabungkan: mulai dari data asli ternormalisasi, isi posisi target & natural missing dgn imputasi
filled = data.copy()
fill_pos = (cond_mask==0)   # semua posisi yang tak jadi kondisi (target + natural)
filled[fill_pos] = imputed_median[fill_pos]

# balik ke [n_ok, n_feat] lalu denormalisasi
flat = filled.reshape(-1, n_feat)[:n_rows]
flat_denorm = flat * stds + means

df_imp = df_support.iloc[:len(flat_denorm)].copy().reset_index(drop=True)
for k, cname in enumerate(FEAT_COLS):
    df_imp[cname] = flat_denorm[:, k]
print('Data terimputasi tersusun:', df_imp.shape)
df_imp.head()

## Cell 12 — Structural Violation Rate (SVR): Dua Skenario

- **Skenario A (tanpa survey rule engine):** keluaran imputasi apa adanya. Model mengisi
  posisi structural yang seharusnya kosong.
- **Skenario B (dengan survey rule engine):** posisi structural dikembalikan kosong via
  restore_structural.

SVR = proporsi posisi structural (yang seharusnya kosong) yang justru terisi.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DEMONSTRASI KUALITATIF SURVEY RULE ENGINE (pengganti metrik SVR)
#
# Rule engine tidak memengaruhi akurasi (MAE/RMSE/CRPS), karena posisi structural
# memang kosong pada data asli sehingga tak pernah menjadi target evaluasi.
# Manfaat rule engine bersifat KUALITATIF: menjaga VALIDITAS keluaran agar sesuai
# aturan kuesioner. Di sini ditunjukkan contoh kasus konkret.
# ═══════════════════════════════════════════════════════════════════════════════
def is_filled(v):
    if pd.isna(v): return False
    if isinstance(v, str) and v.strip() == '': return False
    return True

CHECK_COLS = [col('R705'), col('R706'), col('R707')]
R703C = col('R703_A')

# PENTING: mask structural dihitung dari DATA ASLI (df_support), bukan hasil imputasi.
df_asli = df_support.reset_index(drop=True).copy()
df_prep_asli = sr.prepare_susenas_columns(df_asli)
valid_mask = sr.get_valid_mask(df_prep_asli, sr.SUSENAS_SKIP_RULES)
structural = ~valid_mask
cols_all = list(df_prep_asli.columns)

# Skenario A (tanpa rule): df_imp apa adanya (model mengisi semua)
# Skenario B (dengan rule): structural dikembalikan kosong
df_ruled = sr.restore_structural(df_imp.copy(), df_prep_asli, sr.SUSENAS_SKIP_RULES)

# ── Contoh kasus: RT dengan KRT TIDAK bekerja (R706/R707 seharusnya kosong) ──
# Cari baris yang structural pada R706
jR706 = cols_all.index(col('R706'))
idx_structural = [i for i in range(len(df_asli))
                  if structural[i, jR706] and not is_filled(df_asli.iloc[i][col('R706')])]

print(f'Total RT dengan R706/R707 structural (KRT tidak bekerja): {len(idx_structural):,}')
print()
print('CONTOH KASUS (10 RT pertama dengan KRT tidak bekerja):')
print('Perhatikan R706 (lapangan usaha) & R707 (status kerja):')
print()

rows = []
for i in idx_structural[:10]:
    rows.append({
        'RT': i,
        'R703_A': df_asli.iloc[i][R703C],
        'R705': df_asli.iloc[i][col('R705')],
        'R706_tanpa_rule': df_imp.iloc[i][col('R706')],
        'R706_dgn_rule':   df_ruled.iloc[i][col('R706')],
        'R707_tanpa_rule': df_imp.iloc[i][col('R707')],
        'R707_dgn_rule':   df_ruled.iloc[i][col('R707')],
    })
contoh = pd.DataFrame(rows)
print(contoh.to_string(index=False))
print()
print('INTERPRETASI:')
print('- TANPA rule engine: R706/R707 terisi nilai (mis. kode lapangan usaha),')
print('  padahal KRT tidak bekerja -> keluaran MUSTAHIL menurut aturan kuesioner.')
print('- DENGAN rule engine: R706/R707 dikembalikan kosong (NaN) -> keluaran VALID.')
print()

# ── Ringkasan kuantitatif pendukung (berapa banyak sel diselamatkan) ──
n_struct_cells = 0; n_terisi_tanpa = 0
for v in CHECK_COLS:
    j = cols_all.index(v)
    for i in range(len(df_asli)):
        if structural[i, j] and not is_filled(df_asli.iloc[i][v]):
            n_struct_cells += 1
            if is_filled(df_imp.iloc[i][v]):
                n_terisi_tanpa += 1
print(f'Ringkasan: dari {n_struct_cells:,} sel structural (seharusnya kosong),')
print(f'  tanpa rule engine {n_terisi_tanpa:,} sel salah terisi;')
print(f'  dengan rule engine seluruhnya dikembalikan kosong (valid).')
print()
print(f'Akurasi (tidak terpengaruh rule engine): MAE={mae:.4f} RMSE={rmse:.4f} CRPS={crps:.4f}')

# simpan contoh kasus + data hasil
contoh.to_excel(f'{PROJECT_DIR}/susenas_contoh_kasus_rule.xlsx', index=False)
df_imp.to_csv(f'{PROJECT_DIR}/susenas_imputed_TANPA_rule.csv', index=False)
df_ruled.drop(columns=[c for c in ['_r703_bekerja','_working'] if c in df_ruled.columns]).to_csv(f'{PROJECT_DIR}/susenas_imputed_DENGAN_rule.csv', index=False)
print('\nFile tersimpan: susenas_contoh_kasus_rule.xlsx, susenas_imputed_TANPA_rule.csv, susenas_imputed_DENGAN_rule.csv')


## Cell 13 — Simpan Output Terpisah (dengan vs tanpa rule engine)

In [ ]:
# Output sudah disimpan pada cell demonstrasi kualitatif di atas:
#   - susenas_imputed_TANPA_rule.csv   (imputasi tanpa rule engine)
#   - susenas_imputed_DENGAN_rule.csv  (imputasi dengan rule engine, structural dikosongkan)
#   - susenas_contoh_kasus_rule.xlsx   (tabel contoh kasus untuk skripsi)
print("Selesai. Lihat file output di /kaggle/working/ dan tabel contoh kasus di atas.")